In [2]:
import numpy as np
import math
import matplotlib.pyplot as plt
%matplotlib inline  
#this is used for plot the visualization inside the nodeboot without open the visualization box


In [3]:
class Value:
    def __init__(self,data,_op='',children=(),label=''):
        self.data=data                     
        self._op=_op
        self.backward=lambda:None
        self.grad=0.0
        self.prev=set(children)
        self.label=label

    def __repr__(self):
        return f'value:(data={self.data})'
    
    def __add__(self,other):
        other=other if isinstance(other,Value) else Value(other)

        out=Value(self.data+other.data,'+',(self,other))
        def backward():
            self.grad+=1.0*out.grad  #+= it helps to make the grad more reliable because the grad is explanation of how much it 
                                     #influence the next value if we use same object twice like b=a+a the grad of a is 2 
                                     # because a influence twice of the a value to make b so value need to be 2 
                                     # but a are same object so we make += to add the two object grad instead of replace it

            other.grad+=1.0*out.grad
        out.backward=backward
        return out
    
    def __mul__(self,other):
        other=other if isinstance(other,Value) else Value(other)
        out=Value(self.data*other.data,'*',(self,other))

        def backward():
            other.grad+=self.data*out.grad
            self.grad+=other.data*out.grad

        out.backward=backward()
        return out
    
    def __rmul__(self, other): # other * self
        return self * other

    def __truediv__(self, other): # self / other
        return self * other**-1

    def __neg__(self): # -self
        return self * -1

    def __sub__(self, other): # self - other
        return self + (-other)

    def __radd__(self, other): # other + self
        return self + other
    
    def __pow__(self,other):
        assert isinstance(other, (int, float))
        out=Value(self.data**other,f'**{other}',(self,))

        def backward():
            self.grad=other*(self.data**(other-1))

        out.backward=backward()

        return out
    

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward

        return out
    
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out
    
    def backward(self):
        topo=[]
        visited=set()
        def build_topo(v):
            if v not in visited:
                for child in v.prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad=1.0
        for node in reversed(topo):
            node.backward()



In [6]:
a=Value(2.0)
b=Value(3.0)
c=a+b

In [7]:
pip install graphviz


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot